# Grade 12 · Unit 6 — Feature engineering

> **SYNTHETIC** health table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
FEATURES = ["antenatal_visits", "wealth_quintile", "mother_age"]
X, y = h[FEATURES], h["institutional_delivery"]
print(h.shape, "| share delivering in a facility:", round(float(y.mean()), 3))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cv(df_X):
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    scores = cross_val_score(pipe, df_X, y, cv=5)
    assert len(scores) == 5
    return round(float(scores.mean()), 3)

base = h[["antenatal_visits", "wealth_quintile", "mother_age"]]
print("raw features:", cv(base))

## New features
**Predict:** will each help?

In [ ]:
f2 = base.copy()
f2["visits_ge4"] = (h["antenatal_visits"] >= 4).astype(int)                 # a threshold flag (example)
f3 = f2.assign(visits_x_wealth=h["antenatal_visits"] * h["wealth_quintile"])  # interaction
f4 = pd.concat([f3, pd.get_dummies(h["ecological_belt"], prefix="belt", dtype=int)], axis=1)  # encode a category
print("+ flag:", cv(f2), "| + interaction:", cv(f3), "| + belt dummies:", cv(f4))

**Make:** propose one new feature, say *why* it should help, add it, and report the cross-validated change. Beware: would that feature be known at prediction time?